In [5]:
import os
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt

## Dataset & Preprocessing

In [6]:
from keras.datasets import cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [7]:
y_train = y_train.ravel()
# y_test = y_test.ravel()

In [8]:
rng = np.random.default_rng(42)

train_indices = []
val_indices = []

for label in range(10):
    class_indices = np.where(y_train == label)[0]

    selected_indices = rng.choice(class_indices, size=1000, replace=False)

    train_indices.extend(selected_indices[:800])
    val_indices.extend(selected_indices[800:])

train_indices = np.array(train_indices)
val_indices = np.array(val_indices)

x_train_subset = x_train[train_indices]
y_train_subset = y_train[train_indices]

x_val = x_train[val_indices]
y_val = y_train[val_indices]

In [9]:
x_train_subset = x_train_subset.astype(np.float64).reshape(-1, 3072) / 255.0
x_val = x_val.astype(np.float64).reshape(-1, 3072) / 255.0

print("Training:", x_train_subset.shape, y_train_subset.shape)
print("Validation:", x_val.shape, y_val.shape)

Training: (8000, 3072) (8000,)
Validation: (2000, 3072) (2000,)


In [22]:
class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = np.zeros_like(data)


class ReLU:
    def __init__(self):
        self.x = None

    def __call__(self, x):
        self.x = x
        return np.maximum(x, 0)

    def backward(self, grad):
        return grad * (self.x > 0)


class Sigmoid:
    def __init__(self):
        self.x = None
    
    def __call__(self, x):
        self.x = x
        return (1 / (1 + np.exp(-x)))
    
    def backward(self, grad):
        s = 1 / (1 + np.exp(-self.x))
        return grad * s * (1 - s)


In [34]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = Parameter(np.zeros((out_features, in_features)))
        self.b = Parameter(np.zeros(out_features))
        self.x = None

    def __call__(self, x):
        self.x = x
        return x @ self.W.data.T + self.b.data

    def backward(self, grad):
        self.W.grad += grad.T @ self.x
        self.b.grad += grad.sum(axis=0)
        return grad @ self.W.data


class NeuralNetwork:
    def __init__(
        self, 
        input_features, hidden_features, output_features, 
        activation="relu"
    ):
        self.l1 = Linear(input_features, hidden_features)
        
        if activation == "relu":
            self.a1 = ReLU()
        else:
            self.a1 = Sigmoid()

        self.l2 = Linear(hidden_features, output_features)
        self.layers = [self.l1, self.l2]

    def __call__(self, x):
        return self.l2(self.a1(self.l1(x)))

    def backward(self, grad):
        grad = self.l2.backward(grad)
        grad = self.a1.backward(grad)
        self.l1.backward(grad)

    def parameters(self):
        for layer in self.layers:
            yield layer.W
            yield layer.b


In [24]:
class SGD:
    def __init__(self, model, lr=0.01, momentum=0.0):
        self.params = list(model.parameters())
        self.lr = lr
        self.momentum = momentum
        self.velocity = [np.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        for p in self.params:
            p.grad.fill(0)

    def step(self):
        for p, v in zip(self.params, self.velocity):
            v *= self.momentum
            v += p.grad
            p.data -= self.lr * v

In [21]:

# Example: 10-class categorical cross-entropy
rng = np.random.default_rng(42)
model = NeuralNetwork(4, 16, 10)
optimizer = SGD(model, lr=0.01, momentum=0.9)

x = rng.standard_normal((8, 4))
labels = rng.integers(0, 10, size=8)

logits = model(x)
shifted = logits - logits.max(axis=1, keepdims=True)
exp_logits = np.exp(shifted)
probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)

loss = -np.log(probs[np.arange(len(labels)), labels]).mean()

# Equivalent to loss.backward(): seed the gradient at the output.
grad = probs.copy()
grad[np.arange(len(labels)), labels] -= 1
grad /= len(labels)
model.backward(grad)

optimizer.step()
optimizer.zero_grad()

print("Loss:", loss)


Loss: 2.3025850929940455


In [ ]:
x @ W

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
l1 = Linear(in_features=5, out_features=10)
print(l1(x).shape)

(10,)
